# Explainable Android Malware Detection - XGBoost Baseline

This notebook covers the training and evaluation of the baseline XGBoost model on the `cleaned_dataset.csv`.
XGBoost is chosen for its speed and typically superior performance on tabular datasets compared to standard Random Forests.

In [ ]:
# Step 1: Import necessary libraries
import pandas as pd
import numpy as np
import time
import os
import joblib
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure xgboost is installed: pip install xgboost
try:
    from xgboost import XGBClassifier
except ImportError:
    print("XGBoost is not installed. You can install it using: pip install xgboost")

# Create directories for saving models and data mappings
os.makedirs('models', exist_ok=True)
os.makedirs('data', exist_ok=True)

# Optional: SHAP for explainability (ensure shap is installed: pip install shap)
try:
    import shap
except ImportError:
    print("SHAP is not installed. You can install it using: pip install shap")

### Load the Cleaned Dataset

In [ ]:
# Step 2: Load the dataset
data_path = 'cleaned_dataset.csv'
print(f"Loading data from {data_path}...")
df = pd.read_csv(data_path)

print(f"Dataset shape: {df.shape}")
print(f"Class distribution:\n{df['Label'].value_counts(normalize=True) * 100}")
df.head()

### Data Preprocessing & Splitting
We separate the features from the target variable (`Label`) and split the data into training and testing sets. We preserve `apk_id` to trace back test cases during obfuscation.

In [ ]:
# Step 3: Separate features (X), target (y), and IDs
id_col = 'apk_id'
ids = df[id_col]
X = df.drop(['Label', id_col], axis=1)
y = df['Label']

# Step 4: Train-Test Split (80% training, 20% testing)
# random_state=42 ensures reproducibility
X_train, X_test, y_train, y_test, id_train, id_test = train_test_split(
    X, y, ids, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")

print(f"\nClass distribution in training set (Imbalance Check):\n{y_train.value_counts()}")

# Save the test set ID mapping for later obfuscation matching
test_mapping_path = 'data/xgb_test_set_ids.csv'
pd.DataFrame({'apk_id': id_test, 'label': y_test}).to_csv(test_mapping_path, index=False)
print(f"\nTest set IDs saved to {test_mapping_path}")

### Model Initialization, Training and Persistence
We initialize the XGBoost model using the parameters defined in our pipeline document, and apply `scale_pos_weight` to address the class imbalance.

In [ ]:
# Step 5: Initialize and train the XGBoost Classifier
# Calculate class imbalance weight: sum(negative instances) / sum(positive instances)
imbalance_weight = sum(y_train == 0) / sum(y_train == 1)
print(f"Calculated scale_pos_weight: {imbalance_weight:.4f}\n")

xgb_model = XGBClassifier(
    n_estimators=100,       # Number of boosting rounds
    learning_rate=0.1,      # Step size shrinkage used in update to prevent overfitting
    max_depth=6,            # Maximum depth of a tree
    objective='binary:logistic', # Logistic regression for binary classification
    scale_pos_weight=imbalance_weight, # Counteract class imbalance
    random_state=42,        # Lock randomness for fair comparison later
    n_jobs=-1,              # Use all available CPU cores
    eval_metric='logloss'   # Evaluation metric for the validation data
)

print("Training XGBoost Model...")
start_time = time.time()
xgb_model.fit(X_train, y_train)
training_time = time.time() - start_time
print(f"Training completed in {training_time:.2f} seconds.")

# Step 6: Save the trained model to disk
model_path = 'models/xgb_baseline.pkl'
joblib.dump(xgb_model, model_path)
print(f"Model saved to {model_path}")

# Evaluate model size
size_kb = os.path.getsize(model_path) / 1024
print(f"Model size: {size_kb:.2f} KB")

### Performance Evaluation
Now we evaluate the model on the unseen test set.

In [ ]:
# Step 7: Make predictions and evaluate
print("Evaluating model on test set...")
start_time = time.time()
y_pred = xgb_model.predict(X_test)
y_pred_proba = xgb_model.predict_proba(X_test)[:, 1]
inference_time = time.time() - start_time

# Calculate metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_proba)

print("\n--- Model Performance ---")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")
print(f"ROC-AUC:   {roc_auc:.4f}")
print(f"Inference time for {len(y_test)} samples: {inference_time:.4f} seconds")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Benign (0)', 'Malware (1)']))

### Confusion Matrix Visualization

In [ ]:
# Step 8: Plot the Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Benign', 'Malware'], yticklabels=['Benign', 'Malware'])
plt.ylabel('Actual Label')
plt.xlabel('Predicted Label')
plt.title('XGBoost Confusion Matrix')
plt.show()

### Feature Importance (Basic)
XGBoost provides a built-in method for calculating feature importance (often based on weight or gain). Here we visualize the most important features.

In [ ]:
# Step 9: Plot Top 10 Feature Importances (Built-in)
importances = xgb_model.feature_importances_
indices = np.argsort(importances)[::-1]

top_k = 10
top_features = X.columns[indices][:top_k]
top_importances = importances[indices][:top_k]

plt.figure(figsize=(10,6))
plt.barh(range(top_k), top_importances[::-1], align='center', color='orange')
plt.yticks(range(top_k), top_features[::-1])
plt.xlabel('Feature Importance (Gain)')
plt.title('Top 10 Most Important Features (XGBoost Built-in)')
plt.show()